# Auliya Robusta SNI — HAAR, seed 42

Upload `auliya-robusta-sni-frequency-colab-v1.zip` ke `Coffee_Bean_Detection/bundles/` di Drive bersama. Jalankan tiap arm di sesi GPU terpisah. Saat sesi putus, jalankan ulang notebook arm yang sama pada folder Drive yang sama. Gunakan satu sesi aktif per arm. Paket ini berisi kelompok B dan C.


In [ ]:
from google.colab import drive
from pathlib import Path
import hashlib
import shutil
from tempfile import TemporaryDirectory
from zipfile import ZipFile
import torch

ARM = "HAAR"
PROJECT_OVERRIDE = ""  # Isi path folder bersama jika lokasinya berbeda.
BUNDLE_NAME = "auliya-robusta-sni-frequency-colab-v1.zip"
BUNDLE_SHA = "b8795e189b85a13201e54a562cbe0f42fa7e0f2bfc0eb4b6d04f85d2b4cf57be"

if not torch.cuda.is_available():
    raise RuntimeError("Pilih runtime GPU sebelum menjalankan notebook.")
print("GPU:", torch.cuda.get_device_name(0), flush=True)
drive.mount("/content/drive")
mount = Path("/content/drive")
if PROJECT_OVERRIDE:
    candidates = [Path(PROJECT_OVERRIDE)]
else:
    shared = sorted((mount / ".shortcut-targets-by-id").glob("*/Coffee_Bean_Detection"))
    candidates = [*shared, mount / "MyDrive/Coffee_Bean_Detection"]
    candidates += sorted((mount / "MyDrive").glob("*/Coffee_Bean_Detection"))

PROJECT = next((p for p in candidates if (p / "bundles" / BUNDLE_NAME).is_file()), None)
if PROJECT is None:
    raise FileNotFoundError(f"Upload {BUNDLE_NAME} ke Coffee_Bean_Detection/bundles, atau isi PROJECT_OVERRIDE.")
ARCHIVE = PROJECT / "bundles" / BUNDLE_NAME
OUTPUT = PROJECT / "experiments/auliya-robusta-sni-frequency-v1"
print("Drive project:", PROJECT, flush=True)

# Stage a verified copy on the Colab runtime disk, then reuse it in this session.
WORK = Path("/content") / f"auliya-frequency-{BUNDLE_SHA[:16]}"
marker = WORK / "bundle.sha256"
if not marker.is_file():
    if WORK.exists():
        raise RuntimeError(f"Folder cache tanpa marker: {WORK}. Gunakan runtime baru.")
    with TemporaryDirectory(dir="/content", prefix="auliya-stage-") as tmp:
        stage = Path(tmp)
        local_zip = stage / BUNDLE_NAME
        shutil.copyfile(ARCHIVE, local_zip)
        with local_zip.open("rb") as stream:
            digest = hashlib.file_digest(stream, "sha256").hexdigest()
        if digest != BUNDLE_SHA:
            raise RuntimeError(f"SHA paket berbeda: {digest}")
        decoded = stage / "decoded"
        with ZipFile(local_zip) as bundle:
            bundle.extractall(decoded)
        (decoded / "bundle.sha256").write_text(BUNDLE_SHA, encoding="ascii")
        decoded.rename(WORK)
elif marker.read_text(encoding="ascii") != BUNDLE_SHA:
    raise RuntimeError("Marker cache tidak cocok dengan paket.")
REPO = WORK / "repo"
DATA = WORK / "development"
WEIGHTS = WORK / "weights/yolo26n-seg.pt"
print("Paket kode, bobot, dan data pengembangan siap.", flush=True)


In [ ]:
import importlib.metadata
import subprocess
import sys

try:
    ul_version = importlib.metadata.version("ultralytics")
except importlib.metadata.PackageNotFoundError:
    ul_version = None
if ul_version != "8.4.96":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.96"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(REPO)], check=True)


In [ ]:
from collections import deque
from datetime import datetime, timezone
import json
import os
import time

run_dir = OUTPUT / ARM
run_dir.mkdir(parents=True, exist_ok=True)
log = run_dir / "training.log"
progress = run_dir / "progress.json"
result_path = run_dir / "validation_result.json"
env = os.environ.copy()
env["PYTHONPATH"] = str(REPO / "src")
cmd = [
    sys.executable, "-u", "-m", "coffee_detector.experiments.run_auliya_robusta_frequency",
    "--arm", ARM, "--development", str(DATA), "--weights", str(WEIGHTS),
    "--output", str(OUTPUT), "--device", "0", "--input-bundle-sha256", BUNDLE_SHA,
]
print(f"{ARM} · seed 42 · log: {log}", flush=True)
with log.open("a", encoding="utf-8") as stream:
    stream.write(f"\n--- Session {datetime.now(timezone.utc).isoformat()} ---\n")
    stream.flush()
    with subprocess.Popen(cmd, cwd=REPO, env=env, stdout=stream, stderr=subprocess.STDOUT) as process:
        shown = None
        try:
            while process.poll() is None:
                if progress.is_file():
                    record = json.loads(progress.read_text(encoding="utf-8"))
                    epoch = record["epoch"]
                    if shown != epoch:
                        print(f"{ARM}: checkpoint {epoch}/{record['epochs']}", flush=True)
                        shown = epoch
                time.sleep(8)
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=15)
            except subprocess.TimeoutExpired:
                process.kill()
            raise
        if process.returncode:
            with log.open(encoding="utf-8", errors="replace") as saved:
                print("".join(deque(saved, maxlen=45)))
            raise RuntimeError(f"{ARM} berhenti ({process.returncode}). Log: {log}")
report = json.loads(result_path.read_text(encoding="utf-8"))
assert report["arm"] == ARM and report["test_images_accessed"] is False
print(json.dumps(report["metrics"], indent=2))
print("RESULT:", result_path)
